# Sales Pandas Operations Notebook

This notebook demonstrates a complete set of common Pandas operations using `09_Pandas/data/data_sales.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

possible_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = None
for root in possible_roots:
    if (root / "09_Pandas").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing 09_Pandas folder.")

DATA_PATH = PROJECT_ROOT / "09_Pandas" / "data" / "data_sales.csv"
EXPORTS_DIR = PROJECT_ROOT / "09_Pandas" / "data" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Basic exploration
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nNull values:\n", df.isna().sum())
df.describe(include="all")

## Exercise 1 - Exploration

Try these tasks before moving on:

1. Print the last 8 rows (`tail`).
2. Show only `producto` and `region` for the first 12 rows.
3. Count unique products and regions.

> Suggested methods: `tail`, column selection, `nunique`.

In [ ]:
# Selection and indexing
df[["fecha", "producto", "ventas"]].head(10)

In [ ]:
# Filtering, query, sorting
filtered = df[(df["categoria"] == "Electrónica") & (df["cantidad"] >= 20)]
query_result = df.query("ventas > 200 and cantidad >= 10")
sorted_df = df.sort_values(["region", "ventas"], ascending=[True, False])

print("Filtered rows:", len(filtered))
print("Query rows:", len(query_result))
sorted_df.head(10)

## Exercise 2 - Filtering and Sorting

Practice with boolean masks and `query`:

1. Filter rows where `region == "Norte"` and `ventas > 300`.
2. Find rows where `categoria == "Muebles"` and `cantidad >= 10`.
3. Sort by `cantidad` descending and show top 10 rows.

> Suggested methods: masks, `query`, `sort_values`, `head`.

In [ ]:
# Cleaning and feature engineering
df_clean = df.copy()
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], errors="coerce")
df_clean["ventas"] = pd.to_numeric(df_clean["ventas"], errors="coerce")
df_clean["cantidad"] = pd.to_numeric(df_clean["cantidad"], errors="coerce")
df_clean = df_clean.dropna(subset=["fecha", "ventas", "cantidad"]).copy()
df_clean["ingreso_total"] = df_clean["ventas"] * df_clean["cantidad"]
df_clean["mes"] = df_clean["fecha"].dt.to_period("M").astype(str)

df_clean.head()

In [ ]:
# GroupBy and aggregations
summary_by_product = pd.DataFrame(
    df_clean.groupby("producto", as_index=False).agg(
        total_qty=("cantidad", "sum"),
        total_revenue=("ingreso_total", "sum"),
        avg_price=("ventas", "mean"),
    )
)
summary_by_product_rows = sorted(
    summary_by_product.itertuples(index=False),
    key=lambda row: float(row[2]),
    reverse=True,
)
summary_by_product = pd.DataFrame(
    summary_by_product_rows,
    columns=["producto", "total_qty", "total_revenue", "avg_price"],
)

summary_by_region = pd.DataFrame(
    df_clean.groupby("region", as_index=False).agg(
        total_revenue=("ingreso_total", "sum"),
        transactions=("producto", "count"),
    )
)
summary_by_region_rows = sorted(
    summary_by_region.itertuples(index=False),
    key=lambda row: float(row[1]),
    reverse=True,
)
summary_by_region = pd.DataFrame(
    summary_by_region_rows,
    columns=["region", "total_revenue", "transactions"],
)

summary_by_product.head(10)

## Exercise 3 - Aggregations and Pivot

Build your own summaries:

1. Revenue by `categoria` (descending).
2. Quantity sold by `region`.
3. Pivot table with index=`mes`, columns=`categoria`, values=`ingreso_total`.

> Suggested methods: `groupby`, `agg`, `pivot_table`.

In [ ]:
# Pivot table and time-series view
pivot_region_category = pd.pivot_table(
    df_clean,
    values="ingreso_total",
    index="region",
    columns="categoria",
    aggfunc="sum",
    fill_value=0,
)

monthly_revenue = pd.DataFrame(
    df_clean.groupby("mes", as_index=False).agg(monthly_revenue=("ingreso_total", "sum"))
)

pivot_region_category

In [ ]:
# Visualization
plt.figure(figsize=(8, 4.5))
plt.bar(summary_by_region["region"], summary_by_region["total_revenue"], color="#4C78A8")
plt.title("Total Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
# Export results
summary_by_product.to_csv(EXPORTS_DIR / "sales_summary_by_product.csv", index=False)
summary_by_region.to_csv(EXPORTS_DIR / "sales_summary_by_region.csv", index=False)
monthly_revenue.to_csv(EXPORTS_DIR / "sales_monthly_revenue.csv", index=False)
pivot_region_category.to_csv(EXPORTS_DIR / "sales_pivot_region_category.csv")

print("Exports generated in:", EXPORTS_DIR)
print("- sales_summary_by_product.csv")
print("- sales_summary_by_region.csv")
print("- sales_monthly_revenue.csv")
print("- sales_pivot_region_category.csv")

## Checkpoints (Quick Validation)

Run the next cell to confirm key outputs were generated and have expected structure.

In [ ]:
# Validation checks
assert "ingreso_total" in df_clean.columns
assert summary_by_product.shape[0] > 0
assert summary_by_region.shape[0] > 0
assert monthly_revenue.shape[0] > 0

required_exports = [
    EXPORTS_DIR / "sales_summary_by_product.csv",
    EXPORTS_DIR / "sales_summary_by_region.csv",
    EXPORTS_DIR / "sales_monthly_revenue.csv",
    EXPORTS_DIR / "sales_pivot_region_category.csv",
]

missing = [str(path.name) for path in required_exports if not path.exists()]
if missing:
    print("Missing exports:", missing)
else:
    print("All expected exports are present.")

print("Notebook checkpoints passed.")

## Solutions (Reveal)

Use this section only after attempting the exercises.

Tip: Keep these cells collapsed if you want to preserve the challenge-first flow.

### Solution 1 - Exploration

In [ ]:
# Solution 1
print("Last 8 rows:")
print(df.tail(8))

print("\nFirst 12 rows - producto and region:")
print(df.loc[:11, ["producto", "region"]])

print("\nUnique counts:")
print("Unique products:", df["producto"].nunique())
print("Unique regions:", df["region"].nunique())

### Solution 2 - Filtering and Sorting

In [ ]:
# Solution 2
north_high = df[(df["region"] == "Norte") & (df["ventas"] > 300)]
furniture_qty = df[(df["categoria"] == "Muebles") & (df["cantidad"] >= 10)]
top_qty = df.sort_values("cantidad", ascending=False).head(10)

print("Rows where region='Norte' and ventas>300:", len(north_high))
print(north_high.head(10))

print("\nRows where categoria='Muebles' and cantidad>=10:", len(furniture_qty))
print(furniture_qty.head(10))

print("\nTop 10 rows by cantidad:")
print(top_qty)

### Solution 3 - Aggregations and Pivot

In [ ]:
# Solution 3
revenue_by_category = pd.DataFrame(
    df_clean.groupby("categoria", as_index=False).agg(ingreso_total=("ingreso_total", "sum"))
)
revenue_by_category_rows = sorted(
    revenue_by_category.itertuples(index=False),
    key=lambda row: float(row[1]),
    reverse=True,
)
revenue_by_category = pd.DataFrame(revenue_by_category_rows, columns=["categoria", "ingreso_total"])

qty_by_region = pd.DataFrame(
    df_clean.groupby("region", as_index=False).agg(cantidad=("cantidad", "sum"))
)
qty_by_region_rows = sorted(
    qty_by_region.itertuples(index=False),
    key=lambda row: float(row[1]),
    reverse=True,
)
qty_by_region = pd.DataFrame(qty_by_region_rows, columns=["region", "cantidad"])

pivot_month_category = pd.pivot_table(
    df_clean,
    index="mes",
    columns="categoria",
    values="ingreso_total",
    aggfunc="sum",
    fill_value=0,
)

print("Revenue by category:")
print(revenue_by_category)

print("\nQuantity by region:")
print(qty_by_region)

print("\nPivot (mes x categoria):")
print(pivot_month_category)

## Bonus Exercises Pack (More Real-World Cases)

Great idea: here is a broader set of practical scenarios using the same sales dataset.

These exercises are designed to cover more Pandas patterns used in real projects.

### Exercise 4 - Top product per region

Find the top product by `ingreso_total` in each region.

- Build a grouped table by `region` and `producto`.
- Sort by `region` and revenue descending.
- Keep only the top row per region.

In [ ]:
# Exercise 4 - Starter
region_product = pd.DataFrame(
    df_clean.groupby(["region", "producto"], as_index=False).agg(revenue=("ingreso_total", "sum"))
)

region_product_rows = sorted(
    region_product.itertuples(index=False),
    key=lambda row: (str(row[0]), -float(row[2])),
)
region_product_sorted = pd.DataFrame(region_product_rows, columns=["region", "producto", "revenue"])

top_product_per_region = region_product_sorted.drop_duplicates(subset=["region"], keep="first")

top_product_per_region

### Exercise 5 - Daily revenue with rolling average

Create daily revenue and a 7-day rolling average.

- Group by `fecha` and sum `ingreso_total`.
- Add a `rolling_7d` column.
- Plot both lines.

In [ ]:
# Exercise 5 - Starter
daily = pd.DataFrame(
    df_clean.groupby("fecha", as_index=False).agg(daily_revenue=("ingreso_total", "sum"))
)
daily_rows = sorted(daily.itertuples(index=False), key=lambda row: pd.Timestamp(row[0]))
daily = pd.DataFrame(daily_rows, columns=["fecha", "daily_revenue"])

daily["rolling_7d"] = daily["daily_revenue"].rolling(window=7, min_periods=3).mean()

plt.figure(figsize=(10, 4.5))
plt.plot(daily["fecha"], daily["daily_revenue"], label="Daily", alpha=0.5)
plt.plot(daily["fecha"], daily["rolling_7d"], label="Rolling 7D", linewidth=2)
plt.title("Daily Revenue vs 7-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 6 - Price bands with `cut`

Create price bands for `ventas` and analyze transaction counts by band.

Suggested bands:
- `0-50`
- `50-200`
- `200-500`
- `500+`

In [ ]:
# Exercise 6 - Starter
bins = [0, 50, 200, 500, float("inf")]
labels = ["0-50", "50-200", "200-500", "500+"]

df_bands = df_clean.copy()
df_bands["price_band"] = pd.cut(df_bands["ventas"], bins=bins, labels=labels)

band_counts = pd.DataFrame(
    df_bands.groupby("price_band", as_index=False).agg(transactions=("price_band", "size"))
)

band_counts

### Exercise 7 - Region performance vs target

Create a small target table and compare actual revenue vs target by region.

- Build a target DataFrame manually.
- Merge with region summary.
- Add `gap` and `achievement_pct` columns.

In [ ]:
# Exercise 7 - Starter
targets = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Este", "Oeste"],
        "target_revenue": [280000, 250000, 260000, 255000],
    }
)

region_actual = pd.DataFrame(
    df_clean.groupby("region", as_index=False).agg(actual_revenue=("ingreso_total", "sum"))
)

region_vs_target = region_actual.merge(targets, on="region", how="left")
region_vs_target["gap"] = region_vs_target["actual_revenue"] - region_vs_target["target_revenue"]
region_vs_target["achievement_pct"] = (
    region_vs_target["actual_revenue"] / region_vs_target["target_revenue"] * 100
).round(2)

region_rows = sorted(
    region_vs_target.itertuples(index=False),
    key=lambda row: float(row[4]),
    reverse=True,
)
pd.DataFrame(
    region_rows,
    columns=["region", "actual_revenue", "target_revenue", "gap", "achievement_pct"],
)

### Exercise 8 - Outlier detection by IQR

Detect outliers in `ingreso_total` using IQR method and inspect suspicious rows.

In [ ]:
# Exercise 8 - Starter
q1 = df_clean["ingreso_total"].quantile(0.25)
q3 = df_clean["ingreso_total"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = pd.DataFrame(
    df_clean[(df_clean["ingreso_total"] < lower) | (df_clean["ingreso_total"] > upper)]
)

print("IQR lower:", round(lower, 2), "| upper:", round(upper, 2))
print("Outliers found:", len(outliers))
outliers_view = pd.DataFrame(
    outliers.loc[:, ["fecha", "producto", "region", "ventas", "cantidad", "ingreso_total"]]
)
outliers_view.head(15)

### Exercise 9 - Wide to long reshape

Convert selected columns (`ventas`, `cantidad`, `ingreso_total`) from wide format to long format using `melt`.

In [ ]:
# Exercise 9 - Starter
long_metrics = df_clean.melt(
    id_vars=["fecha", "producto", "categoria", "region"],
    value_vars=["ventas", "cantidad", "ingreso_total"],
    var_name="metric",
    value_name="value",
)

long_metrics.head(20)

### Exercise 10 - Reusable analysis function

Create a reusable function that returns product and region summaries from any sales DataFrame.

Try calling it with `df_clean`.

In [ ]:
# Exercise 10 - Starter
def build_sales_kpis(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    product_kpi = pd.DataFrame(
        dataframe.groupby("producto", as_index=False).agg(
            total_qty=("cantidad", "sum"), total_revenue=("ingreso_total", "sum")
        )
    )
    product_rows = sorted(
        product_kpi.itertuples(index=False),
        key=lambda row: float(row[2]),
        reverse=True,
    )
    product_kpi = pd.DataFrame(product_rows, columns=["producto", "total_qty", "total_revenue"])

    region_kpi = pd.DataFrame(
        dataframe.groupby("region", as_index=False).agg(
            total_revenue=("ingreso_total", "sum"), transactions=("producto", "count")
        )
    )
    region_rows = sorted(
        region_kpi.itertuples(index=False),
        key=lambda row: float(row[1]),
        reverse=True,
    )
    region_kpi = pd.DataFrame(region_rows, columns=["region", "total_revenue", "transactions"])

    return product_kpi, region_kpi

product_kpi_test, region_kpi_test = build_sales_kpis(df_clean)
print("Product KPI shape:", product_kpi_test.shape)
print("Region KPI shape:", region_kpi_test.shape)
region_kpi_test

## Bonus Solutions (Reveal)

Run these only after attempting Exercises 4-10.

### Bonus Solution 4 - Top product per region

In [ ]:
# Bonus Solution 4
region_product_solution = pd.DataFrame(
    df_clean.groupby(["region", "producto"], as_index=False).agg(revenue=("ingreso_total", "sum"))
)

region_product_solution_rows = sorted(
    region_product_solution.itertuples(index=False),
    key=lambda row: (str(row[0]), -float(row[2])),
)
region_product_solution_sorted = pd.DataFrame(
    region_product_solution_rows,
    columns=["region", "producto", "revenue"],
)

top_product_per_region_solution = region_product_solution_sorted.drop_duplicates(
    subset=["region"], keep="first"
)

top_product_per_region_solution

### Bonus Solution 5 - Daily revenue and rolling average

In [ ]:
# Bonus Solution 5
daily_solution = pd.DataFrame(
    df_clean.groupby("fecha", as_index=False).agg(daily_revenue=("ingreso_total", "sum"))
)
daily_solution_rows = sorted(
    daily_solution.itertuples(index=False), key=lambda row: pd.Timestamp(row[0])
)
daily_solution = pd.DataFrame(daily_solution_rows, columns=["fecha", "daily_revenue"])

daily_solution["rolling_7d"] = daily_solution["daily_revenue"].rolling(window=7, min_periods=3).mean()

plt.figure(figsize=(10, 4.5))
plt.plot(daily_solution["fecha"], daily_solution["daily_revenue"], label="Daily", alpha=0.5)
plt.plot(daily_solution["fecha"], daily_solution["rolling_7d"], label="Rolling 7D", linewidth=2)
plt.title("Daily Revenue vs 7-Day Rolling Average (Solution)")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

### Bonus Solution 6 - Price bands

In [ ]:
# Bonus Solution 6
bins_solution = [0, 50, 200, 500, float("inf")]
labels_solution = ["0-50", "50-200", "200-500", "500+"]

df_bands_solution = df_clean.copy()
df_bands_solution["price_band"] = pd.cut(df_bands_solution["ventas"], bins=bins_solution, labels=labels_solution)

band_counts_solution = pd.DataFrame(
    df_bands_solution.groupby("price_band", as_index=False).agg(transactions=("price_band", "size"))
)

band_counts_solution

### Bonus Solution 7 - Region performance vs target

In [ ]:
# Bonus Solution 7
targets_solution = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Este", "Oeste"],
        "target_revenue": [280000, 250000, 260000, 255000],
    }
)

region_actual_solution = pd.DataFrame(
    df_clean.groupby("region", as_index=False).agg(actual_revenue=("ingreso_total", "sum"))
)

region_vs_target_solution = region_actual_solution.merge(targets_solution, on="region", how="left")
region_vs_target_solution["gap"] = (
    region_vs_target_solution["actual_revenue"] - region_vs_target_solution["target_revenue"]
)
region_vs_target_solution["achievement_pct"] = (
    region_vs_target_solution["actual_revenue"] / region_vs_target_solution["target_revenue"] * 100
).round(2)

region_solution_rows = sorted(
    region_vs_target_solution.itertuples(index=False),
    key=lambda row: float(row[4]),
    reverse=True,
)
pd.DataFrame(
    region_solution_rows,
    columns=["region", "actual_revenue", "target_revenue", "gap", "achievement_pct"],
)

### Bonus Solution 8 - IQR outlier detection

In [ ]:
# Bonus Solution 8
q1_solution = df_clean["ingreso_total"].quantile(0.25)
q3_solution = df_clean["ingreso_total"].quantile(0.75)
iqr_solution = q3_solution - q1_solution
lower_solution = q1_solution - 1.5 * iqr_solution
upper_solution = q3_solution + 1.5 * iqr_solution

outliers_solution = pd.DataFrame(
    df_clean[(df_clean["ingreso_total"] < lower_solution) | (df_clean["ingreso_total"] > upper_solution)]
)

print("IQR lower:", round(lower_solution, 2), "| upper:", round(upper_solution, 2))
print("Outliers found:", len(outliers_solution))
outliers_solution_view = pd.DataFrame(
    outliers_solution.loc[:, ["fecha", "producto", "region", "ventas", "cantidad", "ingreso_total"]]
)
outliers_solution_view.head(15)

### Bonus Solution 9 - Wide to long reshape

In [ ]:
# Bonus Solution 9
long_metrics_solution = df_clean.melt(
    id_vars=["fecha", "producto", "categoria", "region"],
    value_vars=["ventas", "cantidad", "ingreso_total"],
    var_name="metric",
    value_name="value",
)

long_metrics_solution.head(20)

### Bonus Solution 10 - Reusable KPI function

In [ ]:
# Bonus Solution 10
def build_sales_kpis_solution(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    product_kpi_solution = pd.DataFrame(
        dataframe.groupby("producto", as_index=False).agg(
            total_qty=("cantidad", "sum"), total_revenue=("ingreso_total", "sum")
        )
    )
    product_solution_rows = sorted(
        product_kpi_solution.itertuples(index=False),
        key=lambda row: float(row[2]),
        reverse=True,
    )
    product_kpi_solution = pd.DataFrame(
        product_solution_rows,
        columns=["producto", "total_qty", "total_revenue"],
    )

    region_kpi_solution = pd.DataFrame(
        dataframe.groupby("region", as_index=False).agg(
            total_revenue=("ingreso_total", "sum"), transactions=("producto", "count")
        )
    )
    region_solution_rows = sorted(
        region_kpi_solution.itertuples(index=False),
        key=lambda row: float(row[1]),
        reverse=True,
    )
    region_kpi_solution = pd.DataFrame(
        region_solution_rows,
        columns=["region", "total_revenue", "transactions"],
    )

    return product_kpi_solution, region_kpi_solution

product_kpi_solution, region_kpi_solution = build_sales_kpis_solution(df_clean)
print("Product KPI shape:", product_kpi_solution.shape)
print("Region KPI shape:", region_kpi_solution.shape)
region_kpi_solution